# 07 테스트 리뷰별 이벤트 예측 + 별점 정제 비교

07번에서는 test 리뷰를 각 모델에 넣었을 때 이벤트성 리뷰인지 먼저 확인하고, 그 예측이 후보 1/2 별점 정제에 어떻게 반영되는지까지 연결해서 확인한다.

전체 모델 4개에 대해 계산은 모두 수행하되, 최종 해석은 제안 모델과 validation 기준 최종 선택 모델 중심으로 진행한다.

- 제안 모델 (`outputs/final_proposed_model.joblib`에 저장된 Hybrid MLP: KcBERT PCA + Metadata)
- 선택 모델 (`outputs/final_selected_model.joblib`에 저장된 validation 기준 선택 모델)

정제 방식은 다음 두 가지다.

1. 후보 1: 이벤트 리뷰로 예측된 리뷰 제외 후 가게별 별점 평균
2. 후보 2: 이벤트 확률 기반 가중 평균

즉, 전체 별점 정제 계산은 `4개 모델 x 2개 후보 = 8개 결과`로 수행한다.

이유: 분류 모델의 예측이 실제 별점 정제 후보 1/2에 어떻게 반영되는지 끝까지 확인하기 위해서다.


## 1. 라이브러리 로드

저장된 모델을 불러오고, test split 기준으로 리뷰별 예측과 가게별 별점 정제를 비교한다.

이유: 저장된 모델과 test split을 다시 불러와 리뷰 단위 예측과 가게 단위 정제를 계산하기 위해서다.


In [1]:
import os

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

pd.set_option('display.max_colwidth', 120)


## 2. test 데이터와 원본 리뷰 데이터 연결

05/06번과 같은 방식으로 test split을 재현한다.

`csv/reviews_embeddings_extract.csv`에서 05/06번과 같은 split을 재현하고, 저장 모델 bundle의 `feature_cols`와 일치하는 raw feature를 그대로 사용한다. `store_name`이 비어 있는 행은 `store_url`을 가게 식별자로 사용한다.

이유: raw feature split을 재현해야 저장 모델 bundle의 feature_cols와 입력 컬럼이 정확히 맞기 때문이다.


In [2]:
SEED = 42
LABEL_COL = 'label'
TEXT_COL = 'cleaned_review_text'
REVIEW_TEXT_COL = 'review_text'
STORE_COL = 'store_name'
STORE_URL_COL = 'store_url'
STORE_ID_COL = 'store_id'
RATING_COL = 'rating'

raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')

train_idx, temp_idx = train_test_split(
    raw_df.index,
    test_size=0.3,
    random_state=SEED,
    stratify=raw_df[LABEL_COL],
)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.5,
    random_state=SEED,
    stratify=raw_df.loc[temp_idx, LABEL_COL],
)

y_test = raw_df.loc[test_idx, LABEL_COL].astype(int)

test_review_df = raw_df.loc[test_idx].copy().reset_index().rename(columns={'index': 'raw_index'})
test_review_df[RATING_COL] = pd.to_numeric(test_review_df[RATING_COL], errors='coerce')
test_review_df[STORE_ID_COL] = test_review_df[STORE_COL].fillna('').astype(str).str.strip()
test_review_df[STORE_ID_COL] = test_review_df[STORE_ID_COL].where(
    test_review_df[STORE_ID_COL].ne(''),
    test_review_df[STORE_URL_COL].fillna('').astype(str),
)
test_review_df[STORE_COL] = test_review_df[STORE_COL].fillna('').astype(str).str.strip()
test_review_df[STORE_COL] = test_review_df[STORE_COL].where(
    test_review_df[STORE_COL].ne(''),
    test_review_df[STORE_ID_COL],
)

text_test = test_review_df[TEXT_COL].fillna('').astype(str)

print('test raw feature shape:', test_review_df.shape)
print('test review shape:', test_review_df.shape)
print('test label 분포:', y_test.value_counts().sort_index().to_dict())
print('test 가게 수:', test_review_df[STORE_ID_COL].nunique())


test raw feature shape: (1327, 784)
test review shape: (1327, 784)
test label 분포: {0: 854, 1: 473}
test 가게 수: 118


/var/folders/wq/j2lqqp7n33dgrwj1r4q0nl9m0000gn/T/ipykernel_75123/1403570102.py:10: DtypeWarning: Columns (0: store_name) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')


## 3. 전체 모델 4개 로드

07번에서 비교할 모델은 다음 네 가지다. `final_selected_model.joblib`과 `final_proposed_model.joblib`은 06번을 실행해 생성한 bundle을 사용한다.

1. TF-IDF + Random Forest
2. KcBERT PCA only + MLP
3. 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)
4. 선택 모델 (`outputs/final_selected_model.joblib`의 validation 기준 선택 모델)

이유: baseline, ablation, 제안 모델 bundle, 최종 선택 모델 bundle을 같은 test 리뷰에서 비교하기 위해서다.


In [3]:
model_specs = [
    {
        'model_key': 'baseline_tfidf_random_forest',
        'model_label': 'TF-IDF + Random Forest',
        'model_role': 'baseline',
        'path': 'outputs/baseline_tfidf_random_forest_model.joblib',
        'input_type': 'text',
    },
    {
        'model_key': 'ablation_mlp_kcbert_pca_only',
        'model_label': 'KcBERT PCA only + MLP',
        'model_role': 'ablation',
        'path': 'outputs/ablation_mlp_kcbert_pca_only_model.joblib',
        'input_type': 'tabular',
    },
    {
        'model_key': 'final_proposed_model',
        'model_label': '제안 모델',
        'model_role': 'proposed',
        'path': 'outputs/final_proposed_model.joblib',
        'input_type': 'tabular',
    },
    {
        'model_key': 'final_selected_model',
        'model_label': '선택 모델',
        'model_role': 'selected',
        'path': 'outputs/final_selected_model.joblib',
        'input_type': 'tabular',
    },
]

missing_paths = [spec['path'] for spec in model_specs if not os.path.exists(spec['path'])]
if missing_paths:
    raise FileNotFoundError(f'필요한 모델 파일이 없습니다: {missing_paths}')

model_bundles = {spec['model_key']: joblib.load(spec['path']) for spec in model_specs}

proposed_bundle = model_bundles['final_proposed_model']
proposed_name = proposed_bundle.get('model_name', 'final_proposed_model')
proposed_feature_set = proposed_bundle.get('feature_set')
proposed_label = f"제안 모델 ({proposed_name}: {proposed_feature_set})" if proposed_feature_set else f"제안 모델 ({proposed_name})"

selected_bundle = model_bundles['final_selected_model']
selected_name = selected_bundle.get('model_name', 'final_selected_model')
selected_feature_set = selected_bundle.get('feature_set')
selected_label = f"선택 모델 ({selected_name}: {selected_feature_set})" if selected_feature_set else f"선택 모델 ({selected_name})"
for spec in model_specs:
    if spec['model_key'] == 'final_proposed_model':
        spec['model_label'] = proposed_label
        spec['input_type'] = proposed_bundle.get('input_type', spec['input_type'])
    if spec['model_key'] == 'final_selected_model':
        spec['model_label'] = selected_label
        spec['input_type'] = selected_bundle.get('input_type', spec['input_type'])

loaded_models = pd.DataFrame([
    {
        'model_key': spec['model_key'],
        'model_label': spec['model_label'],
        'model_role': spec['model_role'],
        'input_type': spec['input_type'],
        'path': spec['path'],
    }
    for spec in model_specs
])
display(loaded_models)


,model_key,model_label,model_role,input_type,path
0,baseline_tfidf_random_forest,TF-IDF + Random Forest,baseline,text,outputs/baseline_tfidf_random_forest_model.joblib
1,ablation_mlp_kcbert_pca_only,KcBERT PCA only + MLP,ablation,tabular,outputs/ablation_mlp_kcbert_pca_only_model.joblib
2,final_proposed_model,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),proposed,tabular,outputs/final_proposed_model.joblib
3,final_selected_model,선택 모델 (ablation_mlp_metadata_only: metadata_only),selected,tabular,outputs/final_selected_model.joblib


## 4. test 리뷰별 이벤트 확률 및 예측 계산

각 모델에서 test 리뷰별 이벤트 확률, threshold 기준 이벤트 예측 여부, 실제 라벨과의 정답 여부를 계산한다. 전체 정답률뿐 아니라 이벤트 리뷰 탐지 목적에 맞는 F1, precision, recall, PR-AUC, ROC-AUC도 함께 계산한다.

여기서부터 리뷰 단위로 `이 리뷰가 이벤트성 리뷰인지 아닌지`, `실제로 맞췄는지`, `후보 1,2에 어떻게 반영되는지`를 확인할 수 있다.

이유: 별점 정제는 리뷰별 이벤트 확률과 threshold 판정에서 시작되므로 먼저 예측 테이블을 만든다. accuracy만 보면 일반 리뷰를 많이 맞추는 모델이 좋아 보일 수 있어, 이벤트 클래스 F1을 함께 확인한다.


In [4]:
def predict_event_probability(spec, bundle):
    model = bundle['model']
    if spec['input_type'] == 'text':
        return model.predict_proba(text_test)[:, 1]

    feature_cols = bundle.get('feature_cols')
    if feature_cols is None:
        raise ValueError(f"{spec['model_label']} bundle에 feature_cols가 없습니다.")
    missing_cols = [col for col in feature_cols if col not in test_review_df.columns]
    if missing_cols:
        raise KeyError(f"{spec['model_label']} 입력 feature가 test raw 데이터에 없습니다: {missing_cols[:10]}")
    return model.predict_proba(test_review_df[feature_cols])[:, 1]


def make_error_type(actual, pred):
    conditions = [
        (actual == 1) & (pred == 1),
        (actual == 0) & (pred == 0),
        (actual == 0) & (pred == 1),
        (actual == 1) & (pred == 0),
    ]
    return np.select(conditions, ['TP', 'TN', 'FP', 'FN'], default='UNKNOWN')


def model_metric_row(model_key, y_true, pred, prob):
    return {
        'model_key': model_key,
        'f1': float(f1_score(y_true, pred)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'pr_auc': float(average_precision_score(y_true, prob)),
        'roc_auc': float(roc_auc_score(y_true, prob)),
    }


prediction_frames = []
metric_rows = []
for spec in model_specs:
    bundle = model_bundles[spec['model_key']]
    threshold = float(bundle.get('best_threshold', bundle.get('default_threshold', 0.5)))
    event_prob = predict_event_probability(spec, bundle)
    event_pred = (event_prob >= threshold).astype(int)
    actual_label = y_test.to_numpy()
    is_correct = actual_label == event_pred
    candidate_2_weight = np.clip(1 - event_prob, 0, 1)

    metric_rows.append(model_metric_row(spec['model_key'], actual_label, event_pred, event_prob))

    prediction_frames.append(pd.DataFrame({
        'raw_index': test_review_df['raw_index'].to_numpy(),
        'store_id': test_review_df[STORE_ID_COL].to_numpy(),
        'store_name': test_review_df[STORE_COL].to_numpy(),
        'store_url': test_review_df[STORE_URL_COL].to_numpy(),
        'rating': test_review_df[RATING_COL].to_numpy(),
        'review_text': test_review_df[REVIEW_TEXT_COL].fillna('').astype(str).to_numpy(),
        'cleaned_review_text': test_review_df[TEXT_COL].fillna('').astype(str).to_numpy(),
        'actual_label': actual_label,
        'actual_label_name': np.where(actual_label == 1, '이벤트 리뷰(1)', '일반 리뷰(0)'),
        'model_key': spec['model_key'],
        'model_label': spec['model_label'],
        'model_role': spec['model_role'],
        'threshold': threshold,
        'event_prob': event_prob,
        'event_pred': event_pred,
        'event_pred_name': np.where(event_pred == 1, '이벤트 예측(1)', '일반 예측(0)'),
        'is_correct': is_correct,
        'error_type': make_error_type(actual_label, event_pred),
        'candidate_1_action': np.where(event_pred == 1, '제외', '포함'),
        'candidate_2_weight': candidate_2_weight,
        'candidate_2_weighted_rating': test_review_df[RATING_COL].to_numpy() * candidate_2_weight,
    }))

prediction_detail = pd.concat(prediction_frames, ignore_index=True)
model_metrics = pd.DataFrame(metric_rows)

expected_rows = len(test_review_df) * len(model_specs)
assert len(prediction_detail) == expected_rows, f'리뷰별 예측 행 수가 맞지 않습니다: {len(prediction_detail)} != {expected_rows}'
assert prediction_detail['is_correct'].equals(prediction_detail['actual_label'].eq(prediction_detail['event_pred']))
np.testing.assert_allclose(
    prediction_detail['candidate_2_weight'].to_numpy(),
    1 - prediction_detail['event_prob'].to_numpy(),
)
assert prediction_detail.loc[prediction_detail['event_pred'] == 1, 'candidate_1_action'].eq('제외').all()
assert prediction_detail.loc[prediction_detail['event_pred'] == 0, 'candidate_1_action'].eq('포함').all()

prediction_summary = (
    prediction_detail.groupby(['model_key', 'model_label', 'model_role', 'threshold'])
    .agg(
        review_count=('event_pred', 'size'),
        event_pred_count=('event_pred', 'sum'),
        event_pred_rate=('event_pred', 'mean'),
        correct_count=('is_correct', 'sum'),
        accuracy=('is_correct', 'mean'),
        mean_event_prob=('event_prob', 'mean'),
    )
    .reset_index()
    .merge(model_metrics, on='model_key', how='left')
    .round({
        'threshold': 4,
        'event_pred_rate': 4,
        'accuracy': 4,
        'mean_event_prob': 4,
        'f1': 4,
        'precision': 4,
        'recall': 4,
        'pr_auc': 4,
        'roc_auc': 4,
    })
)

display(prediction_summary)


,model_key,model_label,model_role,threshold,review_count,event_pred_count,event_pred_rate,correct_count,accuracy,mean_event_prob,f1,precision,recall,pr_auc,roc_auc
0,ablation_mlp_kcbert_pca_only,KcBERT PCA only + MLP,ablation,0.5002,1327,580,0.4371,748,0.5637,0.4736,0.4501,0.4086,0.5011,0.4095,0.5633
1,baseline_tfidf_random_forest,TF-IDF + Random Forest,baseline,0.5018,1327,183,0.1379,859,0.6473,0.3484,0.2866,0.5137,0.1987,0.4382,0.5852
2,final_proposed_model,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),proposed,0.5009,1327,585,0.4408,735,0.5539,0.4733,0.4405,0.3983,0.4926,0.4029,0.5547
3,final_selected_model,선택 모델 (ablation_mlp_metadata_only: metadata_only),selected,0.5011,1327,983,0.7408,601,0.4529,0.5023,0.5014,0.3713,0.7717,0.3563,0.4957


## 5. 리뷰 10개 이벤트 판별 예시

test 리뷰 중 10개를 예시로 뽑아, 제안 모델과 선택 모델이 각각 이벤트성 리뷰로 판단했는지 확인한다.

전체 상세 컬럼은 `main_review_predictions_detail` 변수에 남기고, 화면에는 예시 리뷰만 출력한다.

이유: 전체 표는 크므로 일부 리뷰를 샘플로 보여 예측 결과를 사람이 빠르게 검산하기 위해서다.


In [5]:
review_detail_cols = [
    'model_label',
    'raw_index',
    'store_id',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'actual_label_name',
    'event_pred_name',
    'event_prob',
    'threshold',
    'is_correct',
    'error_type',
    'candidate_1_action',
    'candidate_2_weight',
    'candidate_2_weighted_rating',
]

main_review_predictions = (
    prediction_detail[prediction_detail['model_role'].isin(['proposed', 'selected'])]
    .sort_values(['raw_index', 'model_role'])
    .reset_index(drop=True)
)

main_review_predictions_detail = main_review_predictions[review_detail_cols].round({
    'event_prob': 4,
    'threshold': 4,
    'candidate_2_weight': 4,
    'candidate_2_weighted_rating': 4,
})

EXAMPLE_REVIEW_COUNT = 10
EXAMPLE_REVIEW_SEED = 42
example_raw_indices = (
    test_review_df['raw_index']
    .drop_duplicates()
    .sample(n=min(EXAMPLE_REVIEW_COUNT, test_review_df['raw_index'].nunique()), random_state=EXAMPLE_REVIEW_SEED)
)

main_review_predictions_example = main_review_predictions[
    main_review_predictions['raw_index'].isin(example_raw_indices)
].copy()

example_base = (
    main_review_predictions_example
    .drop_duplicates('raw_index')
    [['raw_index', 'rating', 'review_text', 'actual_label_name']]
    .rename(columns={
        'raw_index': '리뷰번호',
        'rating': '별점',
        'review_text': '리뷰',
        'actual_label_name': '실제',
    })
)

proposed_example = (
    main_review_predictions_example[main_review_predictions_example['model_role'] == 'proposed']
    [['raw_index', 'event_pred_name', 'event_prob', 'is_correct']]
    .rename(columns={
        'raw_index': '리뷰번호',
        'event_pred_name': '제안예측',
        'event_prob': '제안확률',
        'is_correct': '제안정답',
    })
)

selected_example = (
    main_review_predictions_example[main_review_predictions_example['model_role'] == 'selected']
    [['raw_index', 'event_pred_name', 'event_prob', 'is_correct']]
    .rename(columns={
        'raw_index': '리뷰번호',
        'event_pred_name': '선택예측',
        'event_prob': '선택확률',
        'is_correct': '선택정답',
    })
)

main_review_predictions_simple = (
    example_base
    .merge(proposed_example, on='리뷰번호', how='left')
    .merge(selected_example, on='리뷰번호', how='left')
    .sort_values('리뷰번호')
    .round({
        '제안확률': 4,
        '선택확률': 4,
    })
)

print('리뷰 10개 이벤트 판별 예시: 제안 모델 + 선택 모델')
display(main_review_predictions_simple)


리뷰 10개 이벤트 판별 예시: 제안 모델 + 선택 모델


,리뷰번호,별점,리뷰,실제,제안예측,제안확률,제안정답,선택예측,선택확률,선택정답
0,873,5.0,양 많아요! 맛있게 잘 먹었습니다!,이벤트 리뷰(1),일반 예측(0),0.4741,False,일반 예측(0),0.4395,False
1,1115,5.0,맛있게 잘 먹었습니다!\n감사합니다!,일반 리뷰(0),일반 예측(0),0.4309,True,이벤트 예측(1),0.5079,False
2,1310,4.0,사장님 리뷰이벤트 고기 눌렀는데 안왔어요 다음에는 넣어주세요,일반 리뷰(0),일반 예측(0),0.4174,True,일반 예측(0),0.4419,True
3,1593,5.0,맛있어요!!,이벤트 리뷰(1),일반 예측(0),0.4072,False,일반 예측(0),0.4377,False
4,3252,5.0,카야버터랑 말렌카 케이크 둘 다 너무 맛있어오ㅠㅠㅠㅠ,일반 리뷰(0),일반 예측(0),0.2870,True,일반 예측(0),0.4411,True
5,4457,5.0,2월에만 3번 주문했어요 ㅎㅎ\n커피 너무 꼬소하고 캔에 넣어주셔서 좋습니다.\n여기 바스크 치즈케익 너무 좋아하는데 오늘은 변주를 위해 휘낭시에랑 에타를 시켜봅니다.\n미니 바스크 치즈케이크도 지나칠 수 없...,일반 리뷰(0),이벤트 예측(1),0.5200,False,이벤트 예측(1),0.5106,False
6,5032,5.0,파스타 후기가 너무 좋아서 시켰는데 아니나 다를까 너무 탁월한 선택이었어요!!!! 거기다 스테이크는 또 왜이렇게 맛나나요ㅜㅜㅠ 정말 잘먹었습니다,일반 리뷰(0),일반 예측(0),0.4347,True,이벤트 예측(1),0.5145,False
7,5919,5.0,배달도 빠르고 피자도 맛있어요\n슈퍼콤비네이션과 베이컨포테이토 반반인데 자극적이지 않은 맛이라 좋네요\n치즈크러스트로 끝까지 맛있게 잘먹었습니다,이벤트 리뷰(1),이벤트 예측(1),0.5094,True,이벤트 예측(1),0.5672,True
8,7904,5.0,너무 맛있어용,일반 리뷰(0),이벤트 예측(1),0.5815,False,일반 예측(0),0.4378,True
9,7960,5.0,맛있어요~,일반 리뷰(0),일반 예측(0),0.4146,True,이벤트 예측(1),0.5050,False


## 6. 후보 1/2 별점 정제 함수 정의

가게별 원래 평균 별점을 기준으로 두 가지 정제 후보를 적용한다.

- 후보 1: 이벤트 리뷰로 예측된 리뷰를 제외하고 평균 계산
- 후보 2: 이벤트 확률이 높을수록 낮은 가중치를 적용해 평균 계산

이유: 후보 1과 후보 2 계산식을 함수로 분리해 모든 모델에 같은 별점 정제 규칙을 적용하기 위해서다.


In [6]:
def original_store_rating(frame):
    return (
        frame.groupby('store_id')
        .agg(
            store_name=('store_name', 'first'),
            review_count=('rating', 'size'),
            original_mean_rating=('rating', 'mean'),
        )
        .reset_index()
    )


def candidate_1_exclude_predicted_event(frame):
    original = original_store_rating(frame)
    kept = frame[frame['event_pred'] == 0]

    refined = (
        kept.groupby('store_id')
        .agg(
            kept_review_count=('rating', 'size'),
            refined_rating=('rating', 'mean'),
        )
        .reset_index()
    )

    result = original.merge(refined, on='store_id', how='left')
    result['fallback_used'] = result['refined_rating'].isna()
    result['kept_review_count'] = result['kept_review_count'].fillna(0).astype(int)
    result['refined_rating'] = result['refined_rating'].fillna(result['original_mean_rating'])
    return result


def candidate_2_probability_weighted(frame):
    work = frame.copy()
    work['weight'] = work['candidate_2_weight']
    work['weighted_rating'] = work['candidate_2_weighted_rating']

    original = original_store_rating(work)
    weighted = (
        work.groupby('store_id')
        .agg(
            weight_sum=('weight', 'sum'),
            weighted_rating_sum=('weighted_rating', 'sum'),
        )
        .reset_index()
    )
    result = original.merge(weighted, on='store_id', how='left')
    result['fallback_used'] = result['weight_sum'].fillna(0) <= 0
    result['refined_rating'] = np.where(
        result['fallback_used'],
        result['original_mean_rating'],
        result['weighted_rating_sum'] / result['weight_sum'],
    )
    return result


def summarize_refinement(store_result, frame, model_info, candidate_key, candidate_label):
    merged = store_result.copy()
    merged['rating_delta'] = merged['refined_rating'] - merged['original_mean_rating']
    metrics = model_metrics[model_metrics['model_key'] == model_info['model_key']].iloc[0]
    return {
        'model_key': model_info['model_key'],
        'model_label': model_info['model_label'],
        'model_role': model_info['model_role'],
        'candidate': candidate_key,
        'candidate_label': candidate_label,
        'threshold': float(frame['threshold'].iloc[0]),
        'store_count': int(merged['store_id'].nunique()),
        'review_count': int(len(frame)),
        'f1': float(metrics['f1']),
        'precision': float(metrics['precision']),
        'recall': float(metrics['recall']),
        'pr_auc': float(metrics['pr_auc']),
        'roc_auc': float(metrics['roc_auc']),
        'accuracy': float(frame['is_correct'].mean()),
        'event_pred_count': int(frame['event_pred'].sum()),
        'event_pred_rate': float(frame['event_pred'].mean()),
        'original_mean_rating': float(frame['rating'].mean()),
        'refined_mean_rating': float(merged['refined_rating'].mean()),
        'mean_rating_delta': float(merged['rating_delta'].mean()),
        'mean_abs_rating_delta': float(merged['rating_delta'].abs().mean()),
        'max_abs_rating_delta': float(merged['rating_delta'].abs().max()),
        'fallback_store_count': int(merged['fallback_used'].sum()),
    }


## 7. 전체 모델에 후보 1/2 일괄 적용

4개 모델 각각에 후보 1과 후보 2를 모두 적용한다.

이유: 모델마다 이벤트 확률이 다르므로 같은 정제 후보를 일괄 적용해 변화 폭을 비교하기 위해서다.


In [7]:
store_results = []
summary_rows = []

candidate_defs = [
    ('candidate_1_exclude_predicted_event', '후보 1: 이벤트 예측 리뷰 제외', candidate_1_exclude_predicted_event),
    ('candidate_2_probability_weighted', '후보 2: 이벤트 확률 기반 가중 평균', candidate_2_probability_weighted),
]

for spec in model_specs:
    frame = prediction_detail[prediction_detail['model_key'] == spec['model_key']].copy()
    for candidate_key, candidate_label, func in candidate_defs:
        store_result = func(frame)
        store_result['model_key'] = spec['model_key']
        store_result['model_label'] = spec['model_label']
        store_result['model_role'] = spec['model_role']
        store_result['candidate'] = candidate_key
        store_result['candidate_label'] = candidate_label
        store_result['rating_delta'] = store_result['refined_rating'] - store_result['original_mean_rating']
        store_results.append(store_result)
        summary_rows.append(summarize_refinement(store_result, frame, spec, candidate_key, candidate_label))

all_store_results = pd.concat(store_results, ignore_index=True)
refinement_summary = pd.DataFrame(summary_rows)

assert len(refinement_summary) == 8, '4개 모델 x 2개 후보 = 8개 결과여야 한다.'

refinement_summary_display = refinement_summary.round({
    'threshold': 4,
    'f1': 4,
    'precision': 4,
    'recall': 4,
    'pr_auc': 4,
    'roc_auc': 4,
    'accuracy': 4,
    'event_pred_rate': 4,
    'original_mean_rating': 4,
    'refined_mean_rating': 4,
    'mean_rating_delta': 4,
    'mean_abs_rating_delta': 4,
    'max_abs_rating_delta': 4,
})

print('전체 모델 x 정제 후보 결과 수:', len(refinement_summary_display))


전체 모델 x 정제 후보 결과 수: 8


## 8. 리뷰별 예측 결과와 가게별 정제 별점 연결

특정 리뷰를 봤을 때 이 리뷰가 이벤트로 판단됐는지뿐 아니라, 그 판단 이후 해당 가게의 평균 별점이 후보 1/2에서 어떻게 바뀌었는지도 같이 확인한다.

여기서는 이후 계산에 필요한 연결 테이블만 만든다. 리뷰별 화면 출력은 5번 예시 표로 제한한다.

이유: 리뷰 단위 예측과 가게 단위 평균 변화가 연결되어야 정제가 실제로 어떻게 반영됐는지 설명할 수 있다.


In [8]:
candidate_1_store = (
    all_store_results[all_store_results['candidate'] == 'candidate_1_exclude_predicted_event']
    [[
        'model_key',
        'store_id',
        'original_mean_rating',
        'refined_rating',
        'rating_delta',
    ]]
    .rename(columns={
        'original_mean_rating': 'original_store_rating',
        'refined_rating': 'candidate_1_store_refined_rating',
        'rating_delta': 'candidate_1_store_rating_delta',
    })
)

candidate_2_store = (
    all_store_results[all_store_results['candidate'] == 'candidate_2_probability_weighted']
    [[
        'model_key',
        'store_id',
        'refined_rating',
        'rating_delta',
    ]]
    .rename(columns={
        'refined_rating': 'candidate_2_store_refined_rating',
        'rating_delta': 'candidate_2_store_rating_delta',
    })
)

review_refinement_detail = (
    prediction_detail
    .merge(candidate_1_store, on=['model_key', 'store_id'], how='left')
    .merge(candidate_2_store, on=['model_key', 'store_id'], how='left')
)

assert len(review_refinement_detail) == len(prediction_detail)

review_refinement_cols = [
    'model_label',
    'raw_index',
    'store_id',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'actual_label_name',
    'event_pred_name',
    'event_prob',
    'is_correct',
    'error_type',
    'candidate_1_action',
    'candidate_2_weight',
    'original_store_rating',
    'candidate_1_store_refined_rating',
    'candidate_1_store_rating_delta',
    'candidate_2_store_refined_rating',
    'candidate_2_store_rating_delta',
]

simple_refinement_cols = [
    'model_label',
    'rating',
    'review_text',
    'actual_label_name',
    'event_pred_name',
    'event_prob',
    'candidate_1_action',
    'candidate_2_weight',
    'original_store_rating',
    'candidate_1_store_refined_rating',
    'candidate_2_store_refined_rating',
]

simple_refinement_column_names = {
    'model_label': '모델',
    'rating': '리뷰별점',
    'review_text': '리뷰',
    'actual_label_name': '실제',
    'event_pred_name': '예측',
    'event_prob': '이벤트확률',
    'candidate_1_action': '후보1',
    'candidate_2_weight': '후보2가중치',
    'original_store_rating': '기존가게별점',
    'candidate_1_store_refined_rating': '후보1정제별점',
    'candidate_2_store_refined_rating': '후보2정제별점',
}

main_review_refinement_detail = (
    review_refinement_detail[review_refinement_detail['model_role'].isin(['proposed', 'selected'])]
    .sort_values(['raw_index', 'model_role'])
    .reset_index(drop=True)
)

main_review_refinement_full = main_review_refinement_detail[review_refinement_cols].round({
    'event_prob': 4,
    'candidate_2_weight': 4,
    'original_store_rating': 4,
    'candidate_1_store_refined_rating': 4,
    'candidate_1_store_rating_delta': 4,
    'candidate_2_store_refined_rating': 4,
    'candidate_2_store_rating_delta': 4,
})

main_review_refinement_simple = (
    main_review_refinement_detail[simple_refinement_cols]
    .rename(columns=simple_refinement_column_names)
    .round({
        '이벤트확률': 4,
        '후보2가중치': 4,
        '기존가게별점': 4,
        '후보1정제별점': 4,
        '후보2정제별점': 4,
    })
)

print('리뷰별 예측과 가게 정제 별점 연결 테이블 생성 완료:', main_review_refinement_simple.shape)


리뷰별 예측과 가게 정제 별점 연결 테이블 생성 완료: (2654, 11)


## 9. 특정 가게의 전체 평점 정제 검증

test 데이터에서 리뷰 수가 많은 가게 하나를 가져와 후보 1/2 계산이 어떻게 적용되는지 확인한다.

- 후보 1: 이벤트로 예측된 리뷰를 제외한 뒤 평균 계산
- 후보 2: `1 - event_prob`를 가중치로 둔 가중 평균 계산

`STORE_EXAMPLE_ID`에 특정 `store_id`를 넣으면 자동 선택이 아니라 해당 가게를 직접 확인할 수 있다. 자동 선택 시에는 리뷰 수가 많은 가게를 우선 선택하고, 동률이면 별점 변화량이 큰 가게를 선택한다.


In [9]:
STORE_EXAMPLE_ID = None
STORE_EXAMPLE_MODEL_ROLES = ['proposed', 'selected']

store_example_pool = (
    all_store_results[all_store_results['model_role'].isin(STORE_EXAMPLE_MODEL_ROLES)]
    .groupby('store_id')
    .agg(
        store_name=('store_name', 'first'),
        review_count=('review_count', 'max'),
        max_abs_rating_delta=('rating_delta', lambda s: s.abs().max()),
    )
    .reset_index()
    .sort_values(['review_count', 'max_abs_rating_delta'], ascending=[False, False])
    .reset_index(drop=True)
)

if STORE_EXAMPLE_ID is None:
    selected_store_id = store_example_pool.iloc[0]['store_id']
else:
    selected_store_id = STORE_EXAMPLE_ID

store_example_detail = review_refinement_detail[
    review_refinement_detail['model_role'].isin(STORE_EXAMPLE_MODEL_ROLES)
    & review_refinement_detail['store_id'].eq(selected_store_id)
].copy()

if store_example_detail.empty:
    raise ValueError(f'선택한 가게가 test 데이터에 없습니다: {selected_store_id}')

manual_rows = []
for (model_key, model_label), group in store_example_detail.groupby(['model_key', 'model_label']):
    original_mean = group['rating'].mean()
    kept = group[group['event_pred'] == 0]
    candidate_1_manual = original_mean if kept.empty else kept['rating'].mean()
    weight_sum = group['candidate_2_weight'].sum()
    candidate_2_manual = (
        original_mean
        if weight_sum <= 0
        else (group['rating'] * group['candidate_2_weight']).sum() / weight_sum
    )
    first = group.iloc[0]
    candidate_1_delta = candidate_1_manual - original_mean
    candidate_2_delta = candidate_2_manual - original_mean
    manual_rows.append({
        '모델': model_label,
        '리뷰수': len(group),
        '이벤트예측수': int(group['event_pred'].sum()),
        '후보1포함수': int((group['event_pred'] == 0).sum()),
        '기존가게별점': original_mean,
        '후보1정제후별점': candidate_1_manual,
        '후보1별점변화': candidate_1_delta,
        '후보1변화요약': f'{original_mean:.4f} -> {candidate_1_manual:.4f} ({candidate_1_delta:+.4f})',
        '후보2정제후별점': candidate_2_manual,
        '후보2별점변화': candidate_2_delta,
        '후보2변화요약': f'{original_mean:.4f} -> {candidate_2_manual:.4f} ({candidate_2_delta:+.4f})',
        '후보1직접계산': candidate_1_manual,
        '후보1저장값': first['candidate_1_store_refined_rating'],
        '후보1저장값차이': candidate_1_manual - first['candidate_1_store_refined_rating'],
        '후보2직접계산': candidate_2_manual,
        '후보2저장값': first['candidate_2_store_refined_rating'],
        '후보2저장값차이': candidate_2_manual - first['candidate_2_store_refined_rating'],
        '후보2평균가중치': group['candidate_2_weight'].mean(),
    })

store_rating_refinement_example = pd.DataFrame(manual_rows).round({
    '기존가게별점': 4,
    '후보1정제후별점': 4,
    '후보1별점변화': 4,
    '후보2정제후별점': 4,
    '후보2별점변화': 4,
    '후보1직접계산': 4,
    '후보1저장값': 4,
    '후보1저장값차이': 4,
    '후보2직접계산': 4,
    '후보2저장값': 4,
    '후보2저장값차이': 4,
    '후보2평균가중치': 4,
})

store_example_candidate_summary = store_rating_refinement_example[[
    '모델',
    '리뷰수',
    '이벤트예측수',
    '후보1포함수',
    '기존가게별점',
    '후보1정제후별점',
    '후보1별점변화',
    '후보1변화요약',
    '후보2정제후별점',
    '후보2별점변화',
    '후보2변화요약',
    '후보2평균가중치',
]]
store_example_check = store_rating_refinement_example[[
    '모델',
    '후보1직접계산',
    '후보1저장값',
    '후보1저장값차이',
    '후보2직접계산',
    '후보2저장값',
    '후보2저장값차이',
]]

store_name_for_print = store_example_detail['store_name'].iloc[0]
print('선택 가게:', store_name_for_print)
print('store_id:', selected_store_id)
print('선택 기준: 리뷰 수가 많은 가게 우선, 동률이면 별점 변화량이 큰 가게')
print('후보 1/2 정제 전후 별점 변화')
display(store_example_candidate_summary)

print('직접 계산값과 저장된 정제값 검산')
display(store_example_check)


선택 가게: 오성육회연어
store_id: 오성육회연어
선택 기준: 리뷰 수가 많은 가게 우선, 동률이면 별점 변화량이 큰 가게
후보 1/2 정제 전후 별점 변화


,모델,리뷰수,이벤트예측수,후보1포함수,기존가게별점,후보1정제후별점,후보1별점변화,후보1변화요약,후보2정제후별점,후보2별점변화,후보2변화요약,후보2평균가중치
0,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),23,14,9,5.0,5.0,0.0,5.0000 -> 5.0000 (+0.0000),5.0,0.0,5.0000 -> 5.0000 (+0.0000),0.4933
1,선택 모델 (ablation_mlp_metadata_only: metadata_only),23,15,8,5.0,5.0,0.0,5.0000 -> 5.0000 (+0.0000),5.0,0.0,5.0000 -> 5.0000 (+0.0000),0.5036


직접 계산값과 저장된 정제값 검산


,모델,후보1직접계산,후보1저장값,후보1저장값차이,후보2직접계산,후보2저장값,후보2저장값차이
0,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),5.0,5.0,0.0,5.0,5.0,0.0
1,선택 모델 (ablation_mlp_metadata_only: metadata_only),5.0,5.0,0.0,5.0,5.0,0.0


## 10. 메인 비교표: 제안 모델 vs 선택 모델

가게별 별점 정제 결과의 최종 해석은 제안 모델과 선택 모델 중심으로 진행한다.

이유: 최종 발표에서는 모든 모델보다 제안 모델과 선택 모델의 차이를 중심으로 설명하는 것이 명확하기 때문이다.


In [10]:
main_roles = ['proposed', 'selected']
main_comparison = (
    refinement_summary[refinement_summary['model_role'].isin(main_roles)]
    .sort_values(['model_role', 'candidate'])
    .reset_index(drop=True)
)

main_comparison_display = main_comparison[[
    'model_label',
    'candidate_label',
    'threshold',
    'review_count',
    'f1',
    'precision',
    'recall',
    'pr_auc',
    'roc_auc',
    'accuracy',
    'event_pred_count',
    'event_pred_rate',
    'store_count',
    'original_mean_rating',
    'refined_mean_rating',
    'mean_rating_delta',
    'mean_abs_rating_delta',
    'max_abs_rating_delta',
    'fallback_store_count',
]].round({
    'threshold': 4,
    'f1': 4,
    'precision': 4,
    'recall': 4,
    'pr_auc': 4,
    'roc_auc': 4,
    'accuracy': 4,
    'event_pred_rate': 4,
    'original_mean_rating': 4,
    'refined_mean_rating': 4,
    'mean_rating_delta': 4,
    'mean_abs_rating_delta': 4,
    'max_abs_rating_delta': 4,
})

print('메인 비교: 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata) vs validation 기준 선택 모델')
display(main_comparison_display)


메인 비교: 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata) vs validation 기준 선택 모델


,model_label,candidate_label,threshold,review_count,f1,precision,recall,pr_auc,roc_auc,accuracy,event_pred_count,event_pred_rate,store_count,original_mean_rating,refined_mean_rating,mean_rating_delta,mean_abs_rating_delta,max_abs_rating_delta,fallback_store_count
0,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),후보 1: 이벤트 예측 리뷰 제외,0.5009,1327,0.4405,0.3983,0.4926,0.4029,0.5547,0.5539,585,0.4408,118,4.896,4.8670,-0.0120,0.0634,0.8889,0
1,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),후보 2: 이벤트 확률 기반 가중 평균,0.5009,1327,0.4405,0.3983,0.4926,0.4029,0.5547,0.5539,585,0.4408,118,4.896,4.8711,-0.0079,0.0211,0.4165,0
2,선택 모델 (ablation_mlp_metadata_only: metadata_only),후보 1: 이벤트 예측 리뷰 제외,0.5011,1327,0.5014,0.3713,0.7717,0.3563,0.4957,0.4529,983,0.7408,118,4.896,4.8543,-0.0247,0.0835,1.1111,12
3,선택 모델 (ablation_mlp_metadata_only: metadata_only),후보 2: 이벤트 확률 기반 가중 평균,0.5011,1327,0.5014,0.3713,0.7717,0.3563,0.4957,0.4529,983,0.7408,118,4.896,4.8793,0.0003,0.0042,0.0846,0


## 11. 부록 전체 비교표

전체 모델 4개에 후보 2개를 모두 적용한 결과다. 이 표에는 별점 정제 변화량뿐 아니라 test 기준 F1, precision, recall, PR-AUC, ROC-AUC, accuracy도 함께 포함한다.

이 표는 부록/참고용으로 사용하고, 최종 해석은 메인 비교표를 중심으로 한다.

이유: 전체 모델 결과는 검증 근거로 남기되 본문 해석이 복잡해지지 않도록 부록으로 분리한다.


In [11]:
appendix_comparison_display = refinement_summary[[
    'model_label',
    'model_role',
    'candidate_label',
    'threshold',
    'review_count',
    'f1',
    'precision',
    'recall',
    'pr_auc',
    'roc_auc',
    'accuracy',
    'event_pred_count',
    'event_pred_rate',
    'store_count',
    'original_mean_rating',
    'refined_mean_rating',
    'mean_rating_delta',
    'mean_abs_rating_delta',
    'max_abs_rating_delta',
    'fallback_store_count',
]].sort_values(['model_role', 'model_label', 'candidate_label']).reset_index(drop=True).round({
    'threshold': 4,
    'f1': 4,
    'precision': 4,
    'recall': 4,
    'pr_auc': 4,
    'roc_auc': 4,
    'accuracy': 4,
    'event_pred_rate': 4,
    'original_mean_rating': 4,
    'refined_mean_rating': 4,
    'mean_rating_delta': 4,
    'mean_abs_rating_delta': 4,
    'max_abs_rating_delta': 4,
})

print('부록: 전체 모델 4개 x 정제 후보 2개')
display(appendix_comparison_display)


부록: 전체 모델 4개 x 정제 후보 2개


,model_label,model_role,candidate_label,threshold,review_count,f1,precision,recall,pr_auc,roc_auc,accuracy,event_pred_count,event_pred_rate,store_count,original_mean_rating,refined_mean_rating,mean_rating_delta,mean_abs_rating_delta,max_abs_rating_delta,fallback_store_count
0,KcBERT PCA only + MLP,ablation,후보 1: 이벤트 예측 리뷰 제외,0.5002,1327,0.4501,0.4086,0.5011,0.4095,0.5633,0.5637,580,0.4371,118,4.896,4.8659,-0.0131,0.0551,1.1111,0
1,KcBERT PCA only + MLP,ablation,후보 2: 이벤트 확률 기반 가중 평균,0.5002,1327,0.4501,0.4086,0.5011,0.4095,0.5633,0.5637,580,0.4371,118,4.896,4.8674,-0.0116,0.0189,0.5005,0
2,TF-IDF + Random Forest,baseline,후보 1: 이벤트 예측 리뷰 제외,0.5018,1327,0.2866,0.5137,0.1987,0.4382,0.5852,0.6473,183,0.1379,118,4.896,4.8743,-0.0046,0.0193,0.4667,0
3,TF-IDF + Random Forest,baseline,후보 2: 이벤트 확률 기반 가중 평균,0.5018,1327,0.2866,0.5137,0.1987,0.4382,0.5852,0.6473,183,0.1379,118,4.896,4.8758,-0.0032,0.0120,0.1624,0
4,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),proposed,후보 1: 이벤트 예측 리뷰 제외,0.5009,1327,0.4405,0.3983,0.4926,0.4029,0.5547,0.5539,585,0.4408,118,4.896,4.8670,-0.0120,0.0634,0.8889,0
5,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),proposed,후보 2: 이벤트 확률 기반 가중 평균,0.5009,1327,0.4405,0.3983,0.4926,0.4029,0.5547,0.5539,585,0.4408,118,4.896,4.8711,-0.0079,0.0211,0.4165,0
6,선택 모델 (ablation_mlp_metadata_only: metadata_only),selected,후보 1: 이벤트 예측 리뷰 제외,0.5011,1327,0.5014,0.3713,0.7717,0.3563,0.4957,0.4529,983,0.7408,118,4.896,4.8543,-0.0247,0.0835,1.1111,12
7,선택 모델 (ablation_mlp_metadata_only: metadata_only),selected,후보 2: 이벤트 확률 기반 가중 평균,0.5011,1327,0.5014,0.3713,0.7717,0.3563,0.4957,0.4529,983,0.7408,118,4.896,4.8793,0.0003,0.0042,0.0846,0


## 12. 오답 리뷰 샘플

FP/FN 오답 리뷰를 확인한다.

- FP: 일반 리뷰인데 이벤트 리뷰로 예측
- FN: 이벤트 리뷰인데 일반 리뷰로 예측

이유: FP/FN 샘플을 보면 별점 정제가 잘못 작동할 수 있는 실제 리뷰 유형을 확인할 수 있다.


In [12]:
error_samples = (
    review_refinement_detail[
        review_refinement_detail['model_role'].isin(['proposed', 'selected'])
        & review_refinement_detail['error_type'].isin(['FP', 'FN'])
    ]
    .sort_values(['model_role', 'error_type', 'event_prob'], ascending=[True, True, False])
    .groupby(['model_label', 'error_type'], group_keys=False)
    .head(10)
)

error_sample_cols = [
    'model_label',
    'error_type',
    'raw_index',
    'store_name',
    'rating',
    'review_text',
    'cleaned_review_text',
    'actual_label_name',
    'event_pred_name',
    'event_prob',
    'candidate_1_action',
    'candidate_2_weight',
    'original_store_rating',
    'candidate_1_store_refined_rating',
    'candidate_2_store_refined_rating',
]

print('제안 모델/선택 모델 FP/FN 오답 샘플')
display(error_samples[error_sample_cols].round({
    'event_prob': 4,
    'candidate_2_weight': 4,
    'original_store_rating': 4,
    'candidate_1_store_refined_rating': 4,
    'candidate_2_store_refined_rating': 4,
}))


제안 모델/선택 모델 FP/FN 오답 샘플


,model_label,error_type,raw_index,store_name,rating,review_text,cleaned_review_text,actual_label_name,event_pred_name,event_prob,candidate_1_action,candidate_2_weight,original_store_rating,candidate_1_store_refined_rating,candidate_2_store_refined_rating
3442,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),FN,1543,https://s.baemin.com/Lb000iNrcrSW0,5.0,맛있어요👍👍👍,맛있어요,이벤트 리뷰(1),일반 예측(0),0.5000,포함,0.5000,5.0000,5.0000,5.0000
3034,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),FN,5178,난곡반점,5.0,사천탕수육 맛집을 찾았습니다!!\n글구 리뷰로 보내주신 미니짜장밥\n퀄리티 대박입니다~\n볶음밥만큼 아주 맛나요ㅋㅋ\n\n중국집은 이제 여기로 정착하겠습니다!!^^\n👍👍👍👍👍,사천탕수육 맛집을 찾았습니다!! 글구 리뷰로 보내주신 미니짜장밥 퀄리티 대박입니다~ 볶음밥만큼 아주 맛나요ㅋㅋ 중국집은 이제 여기로 정착하겠습니다!!^^,이벤트 리뷰(1),일반 예측(0),0.4996,포함,0.5004,5.0000,5.0000,5.0000
3766,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),FN,4814,이프유캔 Coffee&Dessert 강남본점,5.0,근처에서 촬영하다가 리뷰가 좋길래 시켜봤는데\n리뷰가 좋은건 다 이유가 있는 것 같네요!\n안내해주신 시간 내에 빠르게 도착했구요~\n음료가 담긴 병부터 귀여워서 1차로 좋아했다가\n맛보고 2차 감동..!!💛...,근처에서 촬영하다가 리뷰가 좋길래 시켜봤는데 리뷰가 좋은건 다 이유가 있는 것 같네요! 안내해주신 시간 내에 빠르게 도착했구요~ 음료가 담긴 병부터 귀여워서 1차로 좋아했다가 맛보고 2차 감동..!! 사무실 ...,이벤트 리뷰(1),일반 예측(0),0.4992,포함,0.5008,4.9231,5.0000,4.9536
2967,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),FN,2108,https://s.baemin.com/sO000gAx6rl9w,5.0,오자마자 먹어버려서 사진이 없네용...ㅠㅠ\n뿌링핫도그 넘 맛있고 떡도 완전 쫄깃해요!!\n덜맵게 요청드리니까 맵기도 딱 좋았어용,오자마자 먹어버려서 사진이 없네용...ㅠㅠ 뿌링핫도그 넘 맛있고 떡도 완전 쫄깃해요!! 덜맵게 요청드리니까 맵기도 딱 좋았어용,이벤트 리뷰(1),일반 예측(0),0.4988,포함,0.5012,5.0000,5.0000,5.0000
3894,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),FN,2026,https://s.baemin.com/zY000CmvG66LK,5.0,달다구리 맛있네용☺️,달다구리 맛있네용,이벤트 리뷰(1),일반 예측(0),0.4971,포함,0.5029,4.6000,5.0000,4.7359
3911,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),FN,2749,https://s.baemin.com/Ha000fnAkEtl5,5.0,든든하고 맛있어요! 후레이크?좀더 많으면좋을것같네요,든든하고 맛있어요! 후레이크?좀더 많으면좋을것같네요,이벤트 리뷰(1),일반 예측(0),0.4966,포함,0.5034,4.8000,4.8000,4.7622
2991,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),FN,3370,광어군 오백양,5.0,와 첫 주문이였는데 왜 여기를 이제 알게된건지 한탄스럽네요.\n양도 많고 회도 두껍고 맛있습니다.\n리뷰이벤트 회도 양이 장난 아니에요.\n제가 회 너무 좋아하는데 이제라도 알게된걸 감사하게 생각합니다.\n얼...,와 첫 주문이였는데 왜 여기를 이제 알게된건지 한탄스럽네요. 양도 많고 회도 두껍고 맛있습니다. 리뷰이벤트 회도 양이 장난 아니에요. 제가 회 너무 좋아하는데 이제라도 알게된걸 감사하게 생각합니다. 얼음도 깔...,이벤트 리뷰(1),일반 예측(0),0.4959,포함,0.5041,5.0000,5.0000,5.0000
3267,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),FN,6888,인쌩맥주 수유역점,5.0,다맛있고 좋았어요~~!!,다맛있고 좋았어요~~!!,이벤트 리뷰(1),일반 예측(0),0.4940,포함,0.5060,5.0000,5.0000,5.0000
2686,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),FN,5439,드디어 찾은 인생 최고의 중국집,5.0,헐 짜장면 맛집 발견!! 탕수육도 탕수육인데 짜장이 진짜 맛있어요 요즘 그 면이랑 소스랑 따로노는 거 같은 그 이상한 짜장 말고 진짜 옛날에 먹던 짜장 그 맛..? 너무 좋아요! 포장도 깔끔하게 해주셔서 치우...,헐 짜장면 맛집 발견!! 탕수육도 탕수육인데 짜장이 진짜 맛있어요 요즘 그 면이랑 소스랑 따로노는 거 같은 그 이상한 짜장 말고 진짜 옛날에 먹던 짜장 그 맛..? 너무 좋아요! 포장도 깔끔하게 해주셔서 치우...,이벤트 리뷰(1),일반 예측(0),0.4901,포함,0.5099,4.9333,5.0000,4.9453
3431,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),FN,1659,https://s.baemin.com/48000frmedErp,5.0,색다른 두쫀쿠 맛이 첨이라 시켜봤는데 너무 맛있어요!!! 고소짭짤 또 시킬거같아요 ㅎㅎ,색다른 두쫀쿠 맛이 첨이라 시켜봤는데 너무 맛있어요!!! 고소짭짤 또 시킬거같아요 ㅎㅎ,이벤트 리뷰(1),일반 예측(0),0.4899,포함,0.5101,4.9167,4.8000,4.8664


## 13. 별점 변화가 큰 가게 샘플

제안 모델과 선택 모델에서 별점 변화가 큰 가게를 확인한다.

이유: 가게별 별점 변화가 큰 사례를 보면 정제 로직이 사용자에게 체감될 수 있는 영향을 파악할 수 있다.


In [13]:
top_changed_samples = (
    all_store_results[all_store_results['model_role'].isin(['proposed', 'selected'])]
    .assign(abs_rating_delta=lambda df: df['rating_delta'].abs())
    .sort_values('abs_rating_delta', ascending=False)
    [[
        'model_label',
        'candidate_label',
        'store_id',
        'store_name',
        'review_count',
        'original_mean_rating',
        'refined_rating',
        'rating_delta',
        'fallback_used',
    ]]
    .head(20)
    .round({
        'original_mean_rating': 4,
        'refined_rating': 4,
        'rating_delta': 4,
    })
)

display(top_changed_samples)


,model_label,candidate_label,store_id,store_name,review_count,original_mean_rating,refined_rating,rating_delta,fallback_used
732,선택 모델 (ablation_mlp_metadata_only: metadata_only),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/Vp000ty8vKgb7,https://s.baemin.com/Vp000ty8vKgb7,9,4.1111,3.0000,-1.1111,False
724,선택 모델 (ablation_mlp_metadata_only: metadata_only),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/Ns000gJNQSvNT,https://s.baemin.com/Ns000gJNQSvNT,10,4.3000,3.3333,-0.9667,False
713,선택 모델 (ablation_mlp_metadata_only: metadata_only),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/6V000f312C37V,https://s.baemin.com/6V000f312C37V,9,4.8889,4.0000,-0.8889,False
496,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/Vp000ty8vKgb7,https://s.baemin.com/Vp000ty8vKgb7,9,4.1111,5.0000,0.8889,False
781,선택 모델 (ablation_mlp_metadata_only: metadata_only),후보 1: 이벤트 예측 리뷰 제외,빽보이피자 봉천점,빽보이피자 봉천점,13,4.8462,4.0000,-0.8462,False
503,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/fk000fkrS8JGB,https://s.baemin.com/fk000fkrS8JGB,4,2.7500,3.5000,0.7500,False
752,선택 모델 (ablation_mlp_metadata_only: metadata_only),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/wp000eZswFf3l,https://s.baemin.com/wp000eZswFf3l,14,4.7143,4.0000,-0.7143,False
716,선택 모델 (ablation_mlp_metadata_only: metadata_only),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/Bl000flhKeYEy,https://s.baemin.com/Bl000flhKeYEy,6,4.3333,5.0000,0.6667,False
545,제안 모델 (proposed_hybrid_mlp_04: kcbert_pca+metadata),후보 1: 이벤트 예측 리뷰 제외,빽보이피자 봉천점,빽보이피자 봉천점,13,4.8462,4.3333,-0.5128,False
726,선택 모델 (ablation_mlp_metadata_only: metadata_only),후보 1: 이벤트 예측 리뷰 제외,https://s.baemin.com/Q6000jyZx2LMo,https://s.baemin.com/Q6000jyZx2LMo,10,4.5000,4.0000,-0.5000,False


## 14. 최종 해석

07번에서는 test 리뷰별 이벤트 예측 결과와 가게별 별점 정제 결과를 함께 확인한다.

해석 기준은 다음과 같다.

- 개별 리뷰의 별점 자체를 바꾸는 것이 아니라, 해당 리뷰가 속한 가게의 평균 별점 계산에서 제외되거나 낮은 가중치로 반영된다.
- 후보 1은 이벤트 리뷰로 예측된 리뷰를 제외하므로, threshold와 이벤트 예측 개수의 영향을 크게 받는다.
- 후보 2는 이벤트 확률을 연속적인 가중치로 사용하므로, 후보 1보다 변화가 완만할 수 있다.
- 특정 가게 평점 정제 검증 셀을 통해 후보 1/2 계산 결과가 저장된 정제 별점과 일치하는지 확인할 수 있다.
- 선택 모델은 06번에서 validation F1 기준으로 저장된 모델이며, 모델명과 feature set은 bundle에서 동적으로 읽어 표시한다.
- 제안 모델과 선택 모델이 같을 수 있으므로, 발표/보고서에서는 선택 기준과 중복 여부를 함께 설명한다.
- 발표/보고서에서는 전체 모델 결과를 부록으로 두고, 본문에서는 제안 모델과 validation 기준 선택 모델의 리뷰별 예측 및 후보 1/2 별점 정제 결과를 중심으로 설명한다.

07번 결과를 바탕으로 최종 별점 정제 방식은 다음 중 하나로 정리할 수 있다.

1. 이벤트 리뷰를 강하게 제거하고 싶으면 후보 1을 사용한다.
2. 이벤트 가능성을 부드럽게 반영하고 싶으면 후보 2를 사용한다.
3. 현재 모델의 오탐 가능성을 고려하면 후보 2의 확률 기반 가중 평균이 더 안정적인 정제 방식으로 해석될 수 있다.

이유: 후보 1/2의 장단점과 모델 성능 한계를 함께 정리해야 별점 정제 결과를 과장하지 않을 수 있다.
